In [11]:
# ── Cell 0: Install dependencies ─────────────────────────────────────────────
# Run this cell ONCE before anything else. It installs into the active kernel.
import subprocess, sys

packages = ['plotly', 'nbformat', 'ipywidgets']
for pkg in packages:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        capture_output=True, text=True
    )
    status = 'OK' if result.returncode == 0 else f'FAILED: {result.stderr.strip()}'
    print(f'{pkg:15s} → {status}')

print('\nAll done. Now run Cell 1 (Setup).')

plotly          → OK
nbformat        → OK
ipywidgets      → OK

All done. Now run Cell 1 (Setup).


# SAHPD Visualizer
**Solar-Assisted Heat Pump Dryer — Interactive Simulation Notebook**

| Cell | Content |
|---|---|
| 1 | Setup & imports |
| 2 | **User parameters** — pick config, location, solar area here |
| 3 | Run simulation |
| 4 | System flow diagram |
| 5 | Drying curves (moisture content + drying rate) |
| 6 | Per-tray moisture heatmap |
| 7 | Energy flows + COP |
| 8 | Mode A–E comparison |

In [12]:
# ── Cell 1: Setup ────────────────────────────────────────────────────────────
import sys
from pathlib import Path

# Adjust path so rq1 package is importable
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR          # notebook lives in RQ1/
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from rq1.config_solar_hp import (
    LOCATION_ELEVATIONS_M,
    make_config_A_HP_only,
    make_config_B_solar_HP_series,
    make_config_C_solar_HP_evap,
    make_config_D_solar_only,
    make_config_E_solar_evap_cond_cascade,
)
from rq1.dryer_solar_hp import run_solar_hp_dryer_simulation

# ── Colour palette (one per config) ──────────────────────────────────────────
CONFIG_COLORS = {
    'A': '#1f77b4',  # blue
    'B': '#ff7f0e',  # orange
    'C': '#2ca02c',  # green
    'D': '#d62728',  # red
    'E': '#9467bd',  # purple
}
CONFIG_LABELS = {
    'A': 'A: HP-only',
    'B': 'B: Solar+HP series',
    'C': 'C: Solar HP-evap boost',
    'D': 'D: Solar-only',
    'E': 'E: Solar cascade',
}

print('Setup complete.')

Setup complete.


In [13]:
# ── Cell 2: USER PARAMETERS — edit these ─────────────────────────────────────

CONFIG      = 'B'           # 'A' | 'B' | 'C' | 'D' | 'E'
LOCATION    = 'kathmandu'   # 'kathmandu' | 'biratnagar' | 'dhulikhel'
SOLAR_AREA  = 10.0          # m²  (ignored for Config A)
R_RECIRC    = 0.0           # recirculation ratio 0–1
MAX_HOURS   = 72.0          # simulation time limit [h]

WEATHER_PATH = PROJECT_ROOT / 'data' / 'ambient' / f'{LOCATION}_pvgis_standard.csv'
PHASE2_DIR   = PROJECT_ROOT / 'outputs'
PHASE2_PATH  = PHASE2_DIR / 'phase2c_for_chamber.csv'
ELEV_M       = LOCATION_ELEVATIONS_M.get(LOCATION, 0)

print(f'Config   : {CONFIG}  ({CONFIG_LABELS[CONFIG]})')
print(f'Location : {LOCATION}  ({ELEV_M} m)')
print(f'Solar    : {SOLAR_AREA} m²')
print(f'Weather  : {WEATHER_PATH.name}  (exists: {WEATHER_PATH.exists()})')
print(f'Phase-2  : {PHASE2_PATH.name}   (exists: {PHASE2_PATH.exists()})')

Config   : B  (B: Solar+HP series)
Location : kathmandu  (1350 m)
Solar    : 10.0 m²
Weather  : kathmandu_pvgis_standard.csv  (exists: True)
Phase-2  : phase2c_for_chamber.csv   (exists: True)


In [14]:
# ── Cell 3: Run simulation ───────────────────────────────────────────────────
import importlib, rq1.config_solar_hp as _cfg_mod
importlib.reload(_cfg_mod)

phase2_root = PHASE2_DIR if PHASE2_PATH.exists() else None

_makers = {
    'A': lambda: make_config_A_HP_only(
            ambient_csv=WEATHER_PATH, elevation_m=ELEV_M,
            phase2_root=phase2_root, r_recirc=R_RECIRC),
    'B': lambda: make_config_B_solar_HP_series(
            ambient_csv=WEATHER_PATH, solar_area_m2=SOLAR_AREA,
            elevation_m=ELEV_M, phase2_root=phase2_root, r_recirc=R_RECIRC),
    'C': lambda: make_config_C_solar_HP_evap(
            ambient_csv=WEATHER_PATH, solar_area_m2=SOLAR_AREA,
            elevation_m=ELEV_M, phase2_root=phase2_root),
    'D': lambda: make_config_D_solar_only(
            ambient_csv=WEATHER_PATH, solar_area_m2=SOLAR_AREA,
            elevation_m=ELEV_M, phase2_root=phase2_root),
    'E': lambda: make_config_E_solar_evap_cond_cascade(
            ambient_csv=WEATHER_PATH, solar_area_m2=SOLAR_AREA,
            elevation_m=ELEV_M, phase2_root=phase2_root, r_recirc=R_RECIRC),
}

cfg    = _makers[CONFIG]()
result = run_solar_hp_dryer_simulation(cfg)
df     = result.df.copy()
df['time_h'] = df['time_s'] / 3600.0

print(f'\nResult  : {result.final_message}')
print(f'Rows    : {len(df)}')
print(f'Duration: {df.time_h.max():.1f} h')
print(f'Water removed: {df.m_w_cum_kg.max():.3f} kg')
print(f'Final MR: {df.MR_global.iloc[-1]:.4f}')
if 'COP' in df.columns:
    print(f'Mean COP: {df.COP.replace(0,np.nan).mean():.2f}')


CHAMBER GEOMETRY - B_solar_HP_series
  Location: elevation=1350m, P_atm=86.12kPa, rho_air=0.943kg/m³

PRODUCT:
  Dry mass (m_p_dry_kg):     3.00 kg
  Initial moisture (X0_db):  6.50 kg/kg db
  Fresh apple mass:          22.50 kg
  Water to remove:           19.20 kg

TRAYS:
  Number of trays:           10
  Tray dimensions:           79.1 x 79.1 cm
  Tray area (each):          0.625 m2
  Loading (fresh/tray):      3.6 kg/m2 (actual)
  Air gap:                   12.0 cm

CHAMBER:
  Height:                    166.0 cm
  Length:                    99.1 cm
  Width:                     99.1 cm

AIR FLOW:
  Mass flow rate:            0.0984 kg/s = 354.3 kg/h
  Target velocity:           1.10 m/s
  Actual velocity:           1.10 m/s
  Cross-section area:        0.0949 m2

PRESSURE DROP BREAKDOWN:
  Hydraulic diameter D_h:    208.4 mm
  Reynolds number Re:        10808 (turbulent)
  Darcy friction factor f:   0.0302
  Dynamic pressure q:        0.571 Pa

  Channels (10 gaps):       0.065 Pa/

In [15]:
# ── Cell 4: System Flow Diagram (Plotly 6 compatible) ────────────────────────
snap_idx = len(df) // 2
snap     = df.iloc[snap_idx]
t_snap   = snap['time_h']

def _v(col, fmt='.1f', unit=''):
    if col not in snap.index or pd.isna(snap[col]):
        return '—'
    return f'{snap[col]:{fmt}}{unit}'

# ── All coordinates in data space 0-100 ──────────────────────────────────────
# Each box: (cx, cy, half_w, half_h, fill_color, label_html)
BOXES = {
    'ambient':    (12, 70, 11,  8,  '#D5E8D4',
                   f'<b>Ambient Air</b><br>T={_v("T_amb_C")}°C  RH={_v("RH_amb_pct",".0f")}%'),
    'solar':      (12, 26, 11,  8,  '#FFE6CC',
                   f'<b>Solar Collector</b><br>GHI={_v("GHI_Wm2",".0f")} W/m²<br>'
                   f'Q={_v("Q_solar_cum_kWh",".2f")} kWh'),
    'mix':        (38, 61, 9,   7,  '#DAE8FC',
                   f'<b>Mix / Boost</b>'),
    'condenser':  (63, 70, 10,  7,  '#F8CECC',
                   f'<b>HP Condenser</b><br>T_out={_v("T_to_chamber_C")}°C<br>'
                   f'Q={_v("Q_cond_cum_kWh",".2f")} kWh'),
    'evaporator': (63, 35, 10,  7,  '#D5E8D4',
                   f'<b>HP Evaporator</b><br>T={_v("T_evap_coil_C_dyn")}°C<br>'
                   f'COP={_v("COP",".2f")}'),
    'chamber':    (87, 58, 9,  18,  '#FFF2CC',
                   f'<b>Drying Chamber</b><br>10 trays<br>'
                   f'X={_v("X_db_avg",".3f")} db<br>MR={_v("MR_global",".3f")}'),
    'exhaust':    (87, 88, 9,   5,  '#E1D5E7',
                   f'<b>Exhaust</b><br>T={_v("T_exhaust_C")}°C'),
    'recirc':     (50, 88, 14,  5,  '#F0F0F0',
                   f'<b>Recirculate</b><br>r={_v("r_recirc_actual",".2f")}'),
}

fig = go.Figure()

# ── Draw boxes as shapes ──────────────────────────────────────────────────────
shapes = []
annotations = []

for name, (cx, cy, hw, hh, fill, lbl) in BOXES.items():
    shapes.append(dict(
        type='rect', xref='x', yref='y',
        x0=cx-hw, y0=cy-hh, x1=cx+hw, y1=cy+hh,
        fillcolor=fill, line=dict(color='#555', width=1.5),
    ))
    annotations.append(dict(
        x=cx, y=cy, xref='x', yref='y',
        text=lbl, showarrow=False,
        font=dict(size=9.5), align='center',
        bgcolor='rgba(0,0,0,0)',
    ))

# ── Helper: draw one arrow as scatter trace ───────────────────────────────────
def arrow(x0, y0, x1, y1, label='', lx=None, ly=None, color='#444'):
    fig.add_trace(go.Scatter(
        x=[x0, x1], y=[y0, y1],
        mode='lines+markers',
        line=dict(color=color, width=2),
        marker=dict(symbol='arrow', size=11, color=color, angleref='previous'),
        showlegend=False, hoverinfo='skip',
    ))
    if label:
        fig.add_annotation(
            x=lx if lx else (x0+x1)/2,
            y=ly if ly else (y0+y1)/2,
            xref='x', yref='y',
            text=label, showarrow=False,
            font=dict(size=8.5, color='#333'),
            bgcolor='rgba(255,255,255,0.85)',
            bordercolor='#ccc', borderwidth=1,
        )

# ── Connections ───────────────────────────────────────────────────────────────
# 1. Ambient → Mix
arrow(23, 70, 29, 63,
      label=f'T={_v("T_amb_C")}°C', lx=26, ly=67)

# 2. Solar → Mix
arrow(23, 28, 29, 59,
      label=f'Q_solar', lx=25, ly=45)

# 3. Mix → Condenser
arrow(47, 63, 53, 68,
      label='air', lx=50, ly=64)

# 4. Condenser → Chamber
arrow(73, 70, 78, 65,
      label=f'{_v("T_to_chamber_C")}°C', lx=76, ly=68)

# 5. Chamber → Exhaust (down)
arrow(87, 76, 87, 83,
      label=f'{_v("T_exhaust_C")}°C', lx=90, ly=80)

# 6. Chamber → Recirc (left)
arrow(78, 52, 64, 88,
      label='recirc', lx=69, ly=68)

# 7. Recirc → Mix (left then up)
arrow(36, 88, 35, 68,
      label=f'r={_v("r_recirc_actual",".2f")}', lx=32, ly=79)

# 8. Evaporator ↔ Condenser refrigerant loop
arrow(63, 42, 63, 63,
      label='refrigerant', lx=67.5, ly=52)

fig.update_layout(
    title=dict(
        text=f'SAHPD System — Config {CONFIG}  ({CONFIG_LABELS[CONFIG]})'
             f'<br><sup>Snapshot at t = {t_snap:.1f} h</sup>',
        font=dict(size=13),
    ),
    shapes=shapes,
    annotations=annotations,
    xaxis=dict(range=[0, 100], visible=False),
    yaxis=dict(range=[0, 100], visible=False, scaleanchor='x', scaleratio=0.7),
    height=520,
    margin=dict(l=10, r=10, t=80, b=10),
    plot_bgcolor='white',
)
fig.show()

In [16]:
# ── Cell 5: Drying Curves ────────────────────────────────────────────────────
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=('Moisture Content (dry-basis average)', 'Drying Rate'),
    vertical_spacing=0.10,
)

col = CONFIG_COLORS[CONFIG]

# Top: average moisture content
fig.add_trace(go.Scatter(
    x=df.time_h, y=df.X_db_avg,
    name='X_db avg', line=dict(color=col, width=2),
    hovertemplate='t=%{x:.2f} h<br>X=%{y:.4f} db<extra></extra>',
), row=1, col=1)

# Target moisture line
X_final = cfg.dryer.X_final_db
fig.add_hline(y=X_final, line_dash='dash', line_color='red',
              annotation_text=f'Target X={X_final:.3f}', row=1, col=1)

# Per-tray traces (lighter)
n_trays = 10
for i in range(n_trays):
    col_name = f'X_tray_{i}'
    if col_name in df.columns:
        fig.add_trace(go.Scatter(
            x=df.time_h, y=df[col_name],
            name=f'Tray {i+1}', opacity=0.35,
            line=dict(width=1),
            hovertemplate=f'Tray {i+1}: %{{y:.4f}} db<extra></extra>',
            showlegend=(i == 0),
            legendgroup='trays',
            legendgrouptitle_text='Per-tray' if i == 0 else '',
        ), row=1, col=1)

# Bottom: drying rate (dX/dt proxy via dm_w_total_kg)
fig.add_trace(go.Scatter(
    x=df.time_h, y=df.dm_w_total_kg * 1000,  # g per timestep
    name='dm_w (g/step)', line=dict(color=col, width=2),
    hovertemplate='t=%{x:.2f} h<br>rate=%{y:.3f} g/step<extra></extra>',
), row=2, col=1)

fig.update_yaxes(title_text='Moisture content X [kg/kg db]', row=1, col=1)
fig.update_yaxes(title_text='Water removed per step [g]', row=2, col=1)
fig.update_xaxes(title_text='Time [h]', row=2, col=1)
fig.update_layout(
    title=f'Drying Curves — Config {CONFIG} ({CONFIG_LABELS[CONFIG]})',
    height=560, legend=dict(groupclick='toggleitem'),
)
fig.show()

In [17]:
# ── Cell 6: Per-Tray Moisture Heatmap ───────────────────────────────────────
n_trays = 10
tray_cols = [f'X_tray_{i}' for i in range(n_trays) if f'X_tray_{i}' in df.columns]
n_actual  = len(tray_cols)

Z = df[tray_cols].T.values  # shape: (n_trays, timesteps)

# Downsample time axis for readability (max 300 points)
step = max(1, len(df) // 300)
Z_ds = Z[:, ::step]
t_ds = df.time_h.values[::step]

fig = go.Figure(go.Heatmap(
    z=Z_ds,
    x=t_ds,
    y=[f'Tray {i+1}' for i in range(n_actual)],
    colorscale='RdYlGn_r',
    colorbar=dict(title='X [kg/kg db]'),
    hovertemplate='Tray %{y}<br>t=%{x:.2f} h<br>X=%{z:.4f}<extra></extra>',
))

fig.update_layout(
    title=f'Per-Tray Moisture Content Over Time — Config {CONFIG}',
    xaxis_title='Time [h]',
    yaxis_title='Tray (1=bottom, 10=top)',
    height=400,
)
fig.show()

In [18]:
# ── Cell 7: Energy Flows + COP ───────────────────────────────────────────────
fig = make_subplots(
    rows=2, cols=2, shared_xaxes=False,
    subplot_titles=(
        'Cumulative Energy [kWh]',
        'Instantaneous Temperatures [°C]',
        'COP over time',
        'Solar Irradiance [W/m²]',
    ),
    vertical_spacing=0.14, horizontal_spacing=0.10,
)

# ── Top-left: cumulative energies ────────────────────────────────────────────
energy_traces = [
    ('Q_cond_cum_kWh',  'Condenser Q',  '#d62728'),
    ('Q_solar_cum_kWh', 'Solar Q',      '#ff7f0e'),
    ('W_comp_cum_kWh',  'Compressor W', '#1f77b4'),
    ('W_fan_cum_kWh',   'Fan W',        '#8c564b'),
]
for col_name, label, color in energy_traces:
    if col_name in df.columns:
        fig.add_trace(go.Scatter(
            x=df.time_h, y=df[col_name], name=label,
            line=dict(color=color, width=2),
            hovertemplate=f'{label}: %{{y:.3f}} kWh<extra></extra>',
        ), row=1, col=1)

# ── Top-right: temperatures ───────────────────────────────────────────────────
temp_traces = [
    ('T_amb_C',          'T ambient',        '#aec7e8'),
    ('T_to_chamber_C',   'T to chamber',     '#d62728'),
    ('T_exhaust_C',      'T exhaust',        '#9467bd'),
    ('T_evap_coil_C_dyn','T evap coil',      '#17becf'),
]
for col_name, label, color in temp_traces:
    if col_name in df.columns:
        fig.add_trace(go.Scatter(
            x=df.time_h, y=df[col_name], name=label,
            line=dict(color=color, width=1.5),
            hovertemplate=f'{label}: %{{y:.1f}} °C<extra></extra>',
        ), row=1, col=2)

# ── Bottom-left: COP ─────────────────────────────────────────────────────────
if 'COP' in df.columns:
    cop_clean = df['COP'].replace(0, np.nan)
    fig.add_trace(go.Scatter(
        x=df.time_h, y=cop_clean, name='COP',
        line=dict(color='#2ca02c', width=2),
        hovertemplate='t=%{x:.2f} h<br>COP=%{y:.2f}<extra></extra>',
    ), row=2, col=1)
    mean_cop = cop_clean.mean()
    fig.add_hline(y=mean_cop, line_dash='dot', line_color='green',
                  annotation_text=f'Mean COP={mean_cop:.2f}', row=2, col=1)

# ── Bottom-right: solar irradiance ───────────────────────────────────────────
if 'GHI_Wm2' in df.columns:
    fig.add_trace(go.Scatter(
        x=df.time_h, y=df['GHI_Wm2'], name='GHI',
        fill='tozeroy', fillcolor='rgba(255,200,0,0.2)',
        line=dict(color='#FFA500', width=1.5),
        hovertemplate='t=%{x:.2f} h<br>GHI=%{y:.0f} W/m²<extra></extra>',
    ), row=2, col=2)

fig.update_xaxes(title_text='Time [h]')
fig.update_yaxes(title_text='kWh', row=1, col=1)
fig.update_yaxes(title_text='°C', row=1, col=2)
fig.update_yaxes(title_text='COP [-]', row=2, col=1)
fig.update_yaxes(title_text='W/m²', row=2, col=2)
fig.update_layout(
    title=f'Energy Flows — Config {CONFIG} ({CONFIG_LABELS[CONFIG]})',
    height=600, showlegend=True,
)
fig.show()

In [19]:
# ── Cell 8: Mode A–E Comparison ─────────────────────────────────────────────
# Change COMPARE_CONFIGS list to include whichever modes you want
COMPARE_CONFIGS = ['A', 'B', 'C', 'D', 'E']
COMPARE_SOLAR   = 10.0   # m² — used for B, C, D, E

results_all = {}
for ltr in COMPARE_CONFIGS:
    try:
        cfg_i  = _makers[ltr]()
        res_i  = run_solar_hp_dryer_simulation(cfg_i)
        df_i   = res_i.df.copy()
        df_i['time_h'] = df_i['time_s'] / 3600.0
        results_all[ltr] = df_i
        print(f'Config {ltr}: {res_i.final_message}')
    except Exception as exc:
        print(f'Config {ltr}: FAILED — {exc}')

# ── Subplot: drying curve + SEC bar ─────────────────────────────────────────
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('MR_global vs Time', 'Specific Energy Consumption [kWh/kg water]'),
    column_widths=[0.65, 0.35],
)

sec_data = []
for ltr, df_i in results_all.items():
    col = CONFIG_COLORS[ltr]
    fig.add_trace(go.Scatter(
        x=df_i.time_h, y=df_i.MR_global,
        name=CONFIG_LABELS[ltr], line=dict(color=col, width=2),
        hovertemplate=f'Config {ltr}<br>t=%{{x:.2f}} h<br>MR=%{{y:.4f}}<extra></extra>',
    ), row=1, col=1)

    if 'SEC_elec_kWh_per_kg' in df_i.columns:
        sec_final = df_i['SEC_elec_kWh_per_kg'].dropna()
        sec_val   = sec_final.iloc[-1] if len(sec_final) else np.nan
        sec_data.append((ltr, sec_val, col))

# SEC bar chart
if sec_data:
    ltrs, vals, cols = zip(*sec_data)
    fig.add_trace(go.Bar(
        x=list(ltrs), y=list(vals),
        marker_color=list(cols),
        text=[f'{v:.2f}' for v in vals], textposition='outside',
        name='SEC', showlegend=False,
        hovertemplate='Config %{x}<br>SEC=%{y:.2f} kWh/kg<extra></extra>',
    ), row=1, col=2)

# Target MR line
fig.add_hline(y=0.05, line_dash='dash', line_color='red',
              annotation_text='MR target', row=1, col=1)

fig.update_xaxes(title_text='Time [h]', row=1, col=1)
fig.update_xaxes(title_text='Config', row=1, col=2)
fig.update_yaxes(title_text='Moisture Ratio MR [-]', row=1, col=1)
fig.update_yaxes(title_text='SEC [kWh/kg]', row=1, col=2)
fig.update_layout(
    title=f'Config A–E Comparison — {LOCATION.title()} ({ELEV_M} m), Solar={COMPARE_SOLAR} m²',
    height=480,
)
fig.show()


CHAMBER GEOMETRY - A_HP_only
  Location: elevation=1350m, P_atm=86.12kPa, rho_air=0.943kg/m³

PRODUCT:
  Dry mass (m_p_dry_kg):     3.00 kg
  Initial moisture (X0_db):  6.50 kg/kg db
  Fresh apple mass:          22.50 kg
  Water to remove:           19.20 kg

TRAYS:
  Number of trays:           10
  Tray dimensions:           79.1 x 79.1 cm
  Tray area (each):          0.625 m2
  Loading (fresh/tray):      3.6 kg/m2 (actual)
  Air gap:                   12.0 cm

CHAMBER:
  Height:                    166.0 cm
  Length:                    99.1 cm
  Width:                     99.1 cm

AIR FLOW:
  Mass flow rate:            0.0984 kg/s = 354.3 kg/h
  Target velocity:           1.10 m/s
  Actual velocity:           1.10 m/s
  Cross-section area:        0.0949 m2

PRESSURE DROP BREAKDOWN:
  Hydraulic diameter D_h:    208.4 mm
  Reynolds number Re:        10808 (turbulent)
  Darcy friction factor f:   0.0302
  Dynamic pressure q:        0.571 Pa

  Channels (10 gaps):       0.065 Pa/gap x 10